# Análise dos Resultados da Avaliação do Sistema RAG

Este notebook investiga a qualidade das respostas geradas por um sistema RAG aplicado ao domínio da Lei Geral de Proteção de Dados (LGPD). A análise combina métricas automáticas, avaliação por LLM (judge) e avaliação humana estruturada, aplicadas a 36 pares de perguntas e respostas de um conjunto de teste (gold set).

O objetivo é diagnosticar o comportamento do sistema, identificando padrões de desempenho, principais fontes de erro e limitações do RAG puro no domínio jurídico.

A avaliação considera três dimensões principais:

- Similaridade semântica entre a resposta gerada e o gabarito (Cosine Similarity);
- Avaliação automática por LLM (judge), que examina correção factual, completude e adequação à pergunta;
- Groundedness, que verifica se a resposta está fundamentada nos trechos recuperados, sem introdução de informações externas.

Além das métricas automáticas, foi realizada uma avaliação manual estruturada, permitindo comparar resultados quantitativos com julgamento humano e identificar divergências relevantes.

Com base nesses dados, o notebook explora:

- concordância entre métricas automáticas e avaliação humana;
- desempenho do sistema em diferentes categorias e tipos de pergunta;
- análise das principais origens de erro (retrieval vs. geração);
- comportamento do sistema diante de perguntas fora de escopo e uso de fallback;
- evidências qualitativas de sucesso e falha.

**Objetivo final:** compreender o comportamento do sistema RAG, avaliar sua confiabilidade e identificar oportunidades de melhoria no pipeline de recuperação e geração.

## Carregamento dos dados de avaliação

In [56]:
import pandas as pd

df = pd.read_excel(r"C:\Users\User\Desktop\projeto-rag\rag_chatbot\analysis\evaluation\avaliacao_v2_final.xlsx")

df.head()

,id,categoria,tipo_pergunta,pergunta,resposta_esperada,resposta_modelo,cosine_score,top_k_contexto,artigos_recuperados,artigo_real_recuperado,...,nota_final,classificacao,justificativa,groundedness,justificativa_grounded,avaliacao_manual_qualidade,avaliacao_manual_grounded,origem_erro,controle_fluxo,desempenho_sistema
0,1,conceitos,clara,A LGPD se aplica apenas a dados digitais ou ta...,A LGPD aplica-se ao tratamento de dados pessoa...,A LGPD se aplica ao tratamento de dados pessoa...,0.657291,\n\n[Chunk 1 - Art. 24 - p.11]\ndados a que te...,"Art. 24, Art. 20, Art. 1º, Art. 14, Art. 27, A...","Art. 24, Art. 20, Art. 1º, Art.3, Art. 14, Art...",...,10,correta,"A resposta do modelo está correta, completa e ...",fundamentado,"A resposta está fundamentada no contexto, pois...",correta,fundamentado,nenhum,resposta_direta,satisfatório
1,2,conceitos,clara,Dados de geolocalização são considerados dados...,Dado pessoal é toda informação relacionada a p...,A informação solicitada não foi localizada nos...,0.547151,\n\n[Chunk 1 - Art. 1º - p.0]\nestejam localiz...,"Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...","Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...",...,0,incorreta,A resposta do modelo não aborda a definição de...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,geracao,fallback,insatisfatorio
2,3,conceitos,tecnica,Qual é a distinção jurídica entre as figuras d...,O controlador é quem toma as decisões referent...,O controlador é a pessoa natural ou jurídica q...,0.808525,\n\n[Chunk 1 - Art. 35 - p.14]\norganizacionai...,"Art. 35, Art. 44, Art. 41, Art. 17, Art. 49, A...","Art. 35, Art. 44, Art. 41, Art. 17, Art. 49, A...",...,10,correta,A resposta do modelo está correta em relação a...,fundamentado,"A resposta está fundamentada no contexto, pois...",correta,fundamentado,nenhum,resposta_direta,satisfatório
3,4,conceitos,tecnica,Como o princípio da necessidade deve orientar ...,O tratamento deve limitar-se ao mínimo necessá...,A informação solicitada não foi localizada nos...,0.466143,\n\n[Chunk 1 - Art. 17 - p.9]\nArt. 17. Toda p...,"Art. 17, Art. 12, Art. 24, Art. 27, Artigo não...","Art. 17, Art. 12, Art. 24, Art. 27, Art.5, Art...",...,5,parcial,A resposta do modelo não aborda os princípios ...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,retrieval,fallback,insatisfatorio
4,5,conceitos,ambigua,A LGPD se aplica a uma empresa sediada nos Est...,A aplicação da LGPD independe do país da sede ...,A informação solicitada não foi localizada nos...,0.400113,\n\n[Chunk 1 - Art. 53 - p.19]\nconsiderar o f...,"Art. 53, Art. 55, Art. 24, Art. 44","Art. 53, Art. 55, Art. 24, Art. 44",...,5,parcial,A resposta do modelo não aborda os critérios d...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,retrieval,fallback,insatisfatorio


### 1 - O sistema responde corretamente no geral?

In [57]:
df["avaliacao_manual_qualidade"].value_counts()

avaliacao_manual_qualidade
incorreta    21
correta      13
parcial       2
Name: count, dtype: int64

In [58]:
df["desempenho_sistema"].value_counts()

desempenho_sistema
insatisfatorio    23
satisfatório      13
Name: count, dtype: int64

Interpretação

A maioria das respostas geradas pelo sistema foi considerada incorreta (21/36), com apenas 13 respostas corretas e 2 parciais. Isso se reflete na avaliação geral, em que o desempenho foi classificado como insatisfatório na maior parte dos casos (23). Ou seja, apesar de funcionar bem em algumas perguntas, o sistema ainda não responde de forma confiável na maioria das situações.

### 2 - Em que situações o RAG vai bem ou mal?

In [59]:
# Taxa de sucesso por categoria
taxa_sucesso_categoria = (
    df[df["avaliacao_manual_qualidade"] == "correta"]
    .groupby("categoria")
    .size() / df.groupby("categoria").size()
)

taxa_sucesso_categoria.sort_values(ascending=False)

categoria
fora_de_escopo    1.000000
conceitos         0.333333
direitos          0.333333
bases_legais      0.250000
sancoes           0.200000
governanca        0.166667
dtype: float64

In [60]:
# Taxa por tipo de pergunta
taxa_sucesso_tipo = (
    df[df["avaliacao_manual_qualidade"] == "correta"]
    .groupby("tipo_pergunta")
    .size() / df.groupby("tipo_pergunta").size()
)

taxa_sucesso_tipo.sort_values(ascending=False)

tipo_pergunta
ambigua_juridica        1.000000
juridica_relacionada    1.000000
tecnologia_geral        1.000000
tema_publico            1.000000
clara                   0.500000
tecnica                 0.333333
ambigua                      NaN
incompleta                   NaN
dtype: float64

Interpretação

O sistema apresentou melhor desempenho em perguntas fora de escopo (100%), indicando que o mecanismo de fallback está funcionando corretamente. Entre as categorias jurídicas, o desempenho foi moderado em conceitos e direitos e baixo em bases legais, sanções e governança, sugerindo maior dificuldade em temas mais específicos ou complexos da LGPD.

Quanto ao tipo de pergunta, houve bom desempenho em algumas classes, mas possivelmente com poucos exemplos. Perguntas claras tiveram desempenho mediano, enquanto perguntas técnicas tiveram desempenho baixo. Já perguntas ambíguas ou incompletas não tiveram nenhuma resposta correta, evidenciando dificuldade do sistema quando a consulta é mal formulada ou carece de contexto.

### 3 - O erro começa na busca ou na geração?

In [61]:
# Distribuição da origem do erro
df["origem_erro"].value_counts()

origem_erro
retrieval    17
nenhum       14
geracao       5
Name: count, dtype: int64

In [62]:
# Qualidade × origem do erro
pd.crosstab(
    df["origem_erro"],
    df["avaliacao_manual_qualidade"]
)

avaliacao_manual_qualidade,correta,incorreta,parcial
origem_erro,,,
geracao,0,5,0
nenhum,13,0,1
retrieval,0,16,1


Interpretação

A maioria dos erros do sistema se origina na etapa de retrieval, indicando que o principal problema está na recuperação dos trechos relevantes, e não na geração da resposta. Quando o contexto correto não é recuperado, as respostas tendem a ser incorretas mesmo sem alucinação.

Os erros de geração são menos frequentes, sugerindo que o modelo geralmente responde adequadamente quando recebe contexto apropriado. Já os casos sem erro correspondem quase totalmente a respostas corretas, indicando funcionamento adequado nessas situações.

### 4 - As respostas estão ancoradas na base legal?

In [63]:
df["avaliacao_manual_grounded"].value_counts()

avaliacao_manual_grounded
fundamentado    36
Name: count, dtype: int64

Intrerpretação

Todas as respostas foram consideradas fundamentadas na base legal, indicando ausência de alucinação. O sistema respondeu com base no contexto recuperado, mesmo nos casos em que a resposta foi incorreta ou incompleta. Assim, os erros observados não decorrem de invenção de conteúdo, mas de limitações na recuperação ou interpretação das informações relevantes.

### 5 - O sistema sabe quando não responder?

In [64]:
# Distribuição do controle de fluxo
df["controle_fluxo"].value_counts()

controle_fluxo
fallback           23
resposta_direta    13
Name: count, dtype: int64

In [65]:
# Qualidade × controle de fluxo
pd.crosstab(
    df["controle_fluxo"],
    df["avaliacao_manual_qualidade"]
)

avaliacao_manual_qualidade,correta,incorreta,parcial
controle_fluxo,,,
fallback,5,18,0
resposta_direta,8,3,2


Interpretação

O sistema acionou fallback na maioria dos casos, porém grande parte dessas ocorrências foi avaliada como incorreta, indicando que o mecanismo de não resposta foi utilizado em situações nas quais uma resposta adequada poderia ser fornecida.

Já nas respostas diretas, predominam classificações corretas, sugerindo melhor desempenho quando o sistema decide responder. Em conjunto, os resultados indicam uso frequente e muitas vezes inadequado do fallback, o que reduz a efetividade do sistema.

### 6 - As métricas automáticas refletem qualidade real?

In [66]:
# Coside x Humano
df.groupby("avaliacao_manual_qualidade")["cosine_score"].describe()

,count,mean,std,min,25%,50%,75%,max
avaliacao_manual_qualidade,,,,,,,,
correta,13.0,0.646902,0.204736,0.329469,0.448216,0.748276,0.808525,0.877163
incorreta,21.0,0.476834,0.117681,0.319132,0.416043,0.447291,0.500651,0.770576
parcial,2.0,0.607056,0.240356,0.437099,0.522078,0.607056,0.692035,0.777013


Respostas avaliadas como corretas apresentam, em média, scores de cosine mais altos do que as incorretas, indicando alguma correspondência entre similaridade semântica e qualidade real. No entanto, há sobreposição entre as distribuições, com respostas incorretas também alcançando valores elevados, o que mostra que o cosine capta similaridade textual, mas não garante correção jurídica ou contextual.

In [67]:
# Judge x Humano
pd.crosstab(
    df["nota_final"],
    df["avaliacao_manual_qualidade"],
    normalize="index"
)

avaliacao_manual_qualidade,correta,incorreta,parcial
nota_final,,,
0,0.0000,0.9000,0.1
5,0.3125,0.6875,0.0
10,0.8000,0.1000,0.1


A avaliação automática por judge mostra forte alinhamento com a avaliação humana: notas altas (10) concentram majoritariamente respostas corretas, enquanto notas baixas (0 e 5) estão associadas principalmente a respostas incorretas. Isso indica que o judge reflete melhor a qualidade real das respostas do que métricas puramente semânticas.

In [68]:
#Grounded x humano
pd.crosstab(
    df["groundedness"],
    df["avaliacao_manual_qualidade"],
    normalize="index"
)

avaliacao_manual_qualidade,correta,incorreta,parcial
groundedness,,,
fundamentado,0.361111,0.583333,0.055556


Apesar de todas as respostas terem sido classificadas como fundamentadas, a maioria foi considerada incorreta pela avaliação humana. Isso indica que estar ancorado no contexto recuperado não garante qualidade da resposta, apenas ausência de alucinação. Assim, o sistema tende a errar por usar informações inadequadas ou insuficientes, e não por inventar conteúdo.

### 7 - Exemplos Qualitativos

In [69]:
# sucesso exemplar
df[
    (df["avaliacao_manual_qualidade"] == "correta") &
    (df["origem_erro"] == "nenhum")
].head(1)

,id,categoria,tipo_pergunta,pergunta,resposta_esperada,resposta_modelo,cosine_score,top_k_contexto,artigos_recuperados,artigo_real_recuperado,...,nota_final,classificacao,justificativa,groundedness,justificativa_grounded,avaliacao_manual_qualidade,avaliacao_manual_grounded,origem_erro,controle_fluxo,desempenho_sistema
0,1,conceitos,clara,A LGPD se aplica apenas a dados digitais ou ta...,A LGPD aplica-se ao tratamento de dados pessoa...,A LGPD se aplica ao tratamento de dados pessoa...,0.657291,\n\n[Chunk 1 - Art. 24 - p.11]\ndados a que te...,"Art. 24, Art. 20, Art. 1º, Art. 14, Art. 27, A...","Art. 24, Art. 20, Art. 1º, Art.3, Art. 14, Art...",...,10,correta,"A resposta do modelo está correta, completa e ...",fundamentado,"A resposta está fundamentada no contexto, pois...",correta,fundamentado,nenhum,resposta_direta,satisfatório


In [70]:
# Falha de retrieval
df[df["origem_erro"] == "retrieval"].head(1)

,id,categoria,tipo_pergunta,pergunta,resposta_esperada,resposta_modelo,cosine_score,top_k_contexto,artigos_recuperados,artigo_real_recuperado,...,nota_final,classificacao,justificativa,groundedness,justificativa_grounded,avaliacao_manual_qualidade,avaliacao_manual_grounded,origem_erro,controle_fluxo,desempenho_sistema
3,4,conceitos,tecnica,Como o princípio da necessidade deve orientar ...,O tratamento deve limitar-se ao mínimo necessá...,A informação solicitada não foi localizada nos...,0.466143,\n\n[Chunk 1 - Art. 17 - p.9]\nArt. 17. Toda p...,"Art. 17, Art. 12, Art. 24, Art. 27, Artigo não...","Art. 17, Art. 12, Art. 24, Art. 27, Art.5, Art...",...,5,parcial,A resposta do modelo não aborda os princípios ...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,retrieval,fallback,insatisfatorio


In [71]:
# Falha de geração
df[df["origem_erro"] == "geracao"].head(1)

,id,categoria,tipo_pergunta,pergunta,resposta_esperada,resposta_modelo,cosine_score,top_k_contexto,artigos_recuperados,artigo_real_recuperado,...,nota_final,classificacao,justificativa,groundedness,justificativa_grounded,avaliacao_manual_qualidade,avaliacao_manual_grounded,origem_erro,controle_fluxo,desempenho_sistema
1,2,conceitos,clara,Dados de geolocalização são considerados dados...,Dado pessoal é toda informação relacionada a p...,A informação solicitada não foi localizada nos...,0.547151,\n\n[Chunk 1 - Art. 1º - p.0]\nestejam localiz...,"Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...","Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...",...,0,incorreta,A resposta do modelo não aborda a definição de...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,geracao,fallback,insatisfatorio


In [72]:
# Fallback
df[df["controle_fluxo"] == "fallback"].head(1)

,id,categoria,tipo_pergunta,pergunta,resposta_esperada,resposta_modelo,cosine_score,top_k_contexto,artigos_recuperados,artigo_real_recuperado,...,nota_final,classificacao,justificativa,groundedness,justificativa_grounded,avaliacao_manual_qualidade,avaliacao_manual_grounded,origem_erro,controle_fluxo,desempenho_sistema
1,2,conceitos,clara,Dados de geolocalização são considerados dados...,Dado pessoal é toda informação relacionada a p...,A informação solicitada não foi localizada nos...,0.547151,\n\n[Chunk 1 - Art. 1º - p.0]\nestejam localiz...,"Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...","Art. 1º, Art. 5º, Art. 58, Art. 17, Art. 35, A...",...,0,incorreta,A resposta do modelo não aborda a definição de...,fundamentado,A resposta apenas declara que a informação não...,incorreta,fundamentado,geracao,fallback,insatisfatorio


Interpretação

Foram selecionados exemplos representativos dos principais padrões observados: sucesso sem erro, falha de recuperação, falha de geração e casos de fallback. Não foram identificados comportamentos contraditórios ou classificações ambíguas no conjunto avaliado.

Aproximadamente metade das consultas recupera corretamente a base legal necessária, enquanto a outra metade falha em localizar os dispositivos relevantes, indicando que melhorias no mecanismo de recuperação são essenciais antes de intervenções no modelo gerador.

### 8 - O sistema consegue recuperar os artigos relevantes da LGPD?

In [77]:
# Análise de retrieval apenas quando há artigo esperado
df_in_scope = df[df["retrieval_status"] != "sem_artigo_esperado"]

contagem = df_in_scope["retrieval_status"].value_counts()
percentuais = contagem / len(df_in_scope) * 100

print("Retrieval dentro do escopo:")
print(contagem)
print("\nPercentual (%):")
print(percentuais.round(1))

trouxe_algum = df_in_scope["retrieval_status"].isin(["completo", "parcial"]).sum()
total = len(df_in_scope)

print(f"\nTrouxe pelo menos um artigo correto: {trouxe_algum}/{total} ({trouxe_algum/total:.1%})")

Retrieval dentro do escopo:
retrieval_status
completo      16
inadequado    14
parcial        1
Name: count, dtype: int64

Percentual (%):
retrieval_status
completo      51.6
inadequado    45.2
parcial        3.2
Name: count, dtype: float64

Trouxe pelo menos um artigo correto: 17/31 (54.8%)


Interpretação

Foram excluídas as perguntas fora do escopo normativo da LGPD, nas quais não há artigo específico a ser recuperado. Entre as 31 consultas restantes, o sistema recuperou corretamente todos os artigos necessários em 51,6% dos casos e pelo menos um artigo relevante em 54,8% das consultas. Por outro lado, em 45,2% dos casos nenhum artigo esperado foi recuperado.

Esses resultados indicam que o mecanismo de busca apresenta desempenho moderado: quando o artigo correto é identificado, a recuperação tende a ser completa, porém ainda há uma proporção significativa de falhas em localizar a base legal pertinente.

## Síntese Diagnóstica

- O sistema apresenta bom desempenho em perguntas fora de escopo, demonstrando capacidade de identificar consultas que não pertencem ao domínio da LGPD.

- Dentro do escopo jurídico, o desempenho é limitado. O principal gargalo está na etapa de retrieval: considerando apenas perguntas com base legal esperada, pelo menos um artigo relevante foi recuperado em 54,8% dos casos, enquanto em 45,2% nenhum artigo esperado foi localizado.

- O mecanismo de busca apresenta comportamento polarizado: quando o artigo correto é identificado, a recuperação tende a ser completa; caso contrário, o sistema retorna contexto inadequado.

- Não foram observadas alucinações: todas as respostas se basearam nos trechos recuperados. Entretanto, muitas respostas fundamentadas estavam incorretas devido à recuperação de contexto insuficiente ou irrelevante.

- O fallback é acionado com frequência e nem sempre de forma apropriada, reduzindo a utilidade prática do sistema.

- Entre as métricas automáticas, a avaliação via LLM-as-a-judge mostrou melhor alinhamento com a avaliação humana do que a similaridade semântica.